<a href="https://colab.research.google.com/github/SakinaE/Deep-learning-Neural-networks/blob/main/stock_market_d%26n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

data = pd.read_csv("/content/Market.csv.zip")
data = data[['Close']]


scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(data)


X = []
y = []

for i in range(60, len(scaled_data)):
    X.append(scaled_data[i-60:i, 0])
    y.append(scaled_data[i, 0])

X, y = np.array(X), np.array(y)


X = np.reshape(X, (X.shape[0], X.shape[1], 1))


train_size = int(0.7 * len(X))
val_size = int(0.15 * len(X))

X_train = X[:train_size]
y_train = y[:train_size]

X_val = X[train_size:train_size+val_size]
y_val = y[train_size:train_size+val_size]

X_test = X[train_size+val_size:]
y_test = y[train_size+val_size:]


model = Sequential()

model.add(LSTM(50, return_sequences=True, input_shape=(X.shape[1], 1)))
model.add(Dropout(0.2))

model.add(LSTM(50))
model.add(Dropout(0.2))

model.add(Dense(1, activation='linear'))

optimizer = Adam(learning_rate=0.0001)
model.compile(optimizer=optimizer, loss='mean_squared_error')

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32
)


predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions)


y_test_actual = scaler.inverse_transform(y_test.reshape(-1,1))

rmse = math.sqrt(mean_squared_error(y_test_actual, predictions))
print("RMSE:", rmse)


last_60_days = scaled_data[-60:]
last_60_days = last_60_days.reshape(1, 60, 1)

next_day_prediction = model.predict(last_60_days)
next_day_prediction = scaler.inverse_transform(next_day_prediction)

print("Next Day Prediction:", next_day_prediction[0][0])


future_steps = 5
future_predictions = []

current_input = last_60_days.copy()

for _ in range(future_steps):
    next_pred = model.predict(current_input)[0][0]
    future_predictions.append(next_pred)

    current_input = np.append(current_input[:,1:,:], [[[next_pred]]], axis=1)

future_predictions = scaler.inverse_transform(
    np.array(future_predictions).reshape(-1,1)
)

print("Next 5 Days Forecast:")
for i, val in enumerate(future_predictions):
    print(f"Day {i+1}: {val[0]}")


plt.figure(figsize=(10,5))

plt.plot(data.values, label="Actual")

test_start = train_size + val_size + 60
plt.plot(range(test_start, test_start + len(predictions)), predictions, label="Predicted")

plt.legend()
plt.title("Actual vs Predicted Prices")
plt.show()

print("\nBusiness Insight:")
if future_predictions[-1] > future_predictions[0]:
    print("Upward trend detected → Possible price increase → Consider buying strategy.")
else:
    print("Downward trend detected → Possible price decrease → Consider selling or caution.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
2459/2459 ━━━━━━━━━━━━━━━━━━━━ 154s 61ms/step - loss: nan - val_loss: nan
Epoch 2/20
2459/2459 ━━━━━━━━━━━━━━━━━━━━ 151s 62ms/step - loss: nan - val_loss: nan
Epoch 3/20
2459/2459 ━━━━━━━━━━━━━━━━━━━━ 148s 60ms/step - loss: nan - val_loss: nan
Epoch 4/20
2459/2459 ━━━━━━━━━━━━━━━━━━━━ 203s 61ms/step - loss: nan - val_loss: nan
Epoch 5/20
2459/2459 ━━━━━━━━━━━━━━━━━━━━ 204s 61ms/step - loss: nan - val_loss: nan
Epoch 6/20
2459/2459 ━━━━━━━━━━━━━━━━━━━━ 201s 61ms/step - loss: nan - val_loss: nan
Epoch 7/20
2459/2459 ━━━━━━━━━━━━━━━━━━━━ 198s 60ms/step - loss: nan - val_loss: nan
Epoch 8/20
2459/2459 ━━━━━━━━━━━━━━━━━━━━ 148s 60ms/step - loss: nan - val_loss: nan
Epoch 9/20
2459/2459 ━━━━━━━━━━━━━━━━━━━━ 202s 60ms/step - loss: nan - val_loss: nan
Epoch 10/20
2459/2459 ━━━━━━━━━━━━━━━━━━━━ 202s 60ms/step - loss: nan - val_loss: nan
Epoch 11/20
2459/2459 ━━━━━━━━━━━━━━━━━━━━ 202s 60ms/step - loss: nan - val_loss: nan
Epoch 12/20
2459/2459 ━━━━━━━━━━━━━━━━━━━━ 149s 61ms/step - los